In [5]:
from neo4j import GraphDatabase
from typing import List

class Neo4jRetriever:
    def __init__(self, uri: str, user: str, password: str, database="neo4j"):
        self.driver = GraphDatabase.driver(uri, auth=(user, password))
        self.database = database

    def get_candidates(self, anchor_text: str, source: str, limit: int = 5) -> List[str]:
        query = """
        MATCH (ent:Entity {source: $source})
        WHERE toLower($text) CONTAINS toLower(ent.name)
        MATCH (ent)<-[:HAS_ACTOR|HAS_ACTION|HAS_OBJECT]-(r:Requirement {source: $source})
        WHERE r.text <> $text
        RETURN DISTINCT r.text AS candidate_text
        LIMIT $k
        """

        with self.driver.session(database=self.database) as session:
            res = session.run(
                query,
                text=anchor_text,
                source=source,
                k=limit
            )
            return [r["candidate_text"] for r in res if r["candidate_text"]]

In [6]:
aura_kag = Neo4jRetriever(
            uri="neo4j+s://d1df8710.databases.neo4j.io",
            user="neo4j",
            password="xJ6GyOib4r5d2uq8pkj7QrqNtYSyQRPR_h1l8HiFvRc"
        )

In [7]:
local_kag = Neo4jRetriever(
            uri="neo4j://localhost:7687",
            user="neo4j",
            password="Shreyas@1411"
        )

In [8]:
aura_candidates = aura_kag.get_candidates("TheAviary is a system of systems consisting of a pilot controller (known as the Bird Feeder), a UAV (known as theHummingbird), and up to 4 Viewers (apps running on smart phones or computers).", source="IBM_UAV", limit=15)
print("Hosted Neo4J Candidates:",aura_candidates)

local_candidates = local_kag.get_candidates("TheAviary is a system of systems consisting of a pilot controller (known as the Bird Feeder), a UAV (known as theHummingbird), and up to 4 Viewers (apps running on smart phones or computers).", source="IBM_UAV", limit=15)
print("Local Neo4J Candidates:",local_candidates)

Hosted Neo4J Candidates: ['These are the customer requirements for the Aviary System of Systems.', 'The Aviary shall provide a remote surveillance from the air of a location within 20 miles of the origin.', 'Remote surveillance shall include video streaming for manual navigation of the surveillance platform and for remote viewer by at least four independent operations personnel.', 'The Aviary shall support autonomous mission plans as well as Pilot-directed flight.', 'The Pilot shall be able to fly the UAV to any accessible-by-air location within 20 miles of its origin', 'The Pilot shall communicate with the UAV using custom communications network capable of communicating within a 50-mile radius of the pilot station.', 'All Pilot communication with the UAV shall require a secure connections and shall be protected against snooping and other forms of unauthorized access.', 'The Pilot shall be able to control the speed of flight from 0 to maximum speed in any direction in a smooth fashion.